# K1 — Structure and demography

This page checks phase K1 of the port: the Kiribati model's compartment map, ageing, care
cascade, deaths and births, built in summer4, reproduce the original summer2gen model when
there is no transmission.

**Claim to check.** With `raw_transmission_rate = 0` and homogeneous mixing (golden fixture
`no_transmission`), every one of the 160 compartments matches the original at every year
1850–2035 to a relative error of 1e-5 (absolute floor 1e-6 persons), and the 1850 row is
identical.

The goldens were produced by the original code (`pixi run -e reference golden`); this page
only reads them.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
import sys
import yaml

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from kiribati_tb.demography import AGE_GROUPS, load_demography
from kiribati_tb.model import AGE, compile_model, pmap, run_compartments
from kiribati_tb.paths import GOLDEN, REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "tests"))
from parity import parity_report, read_golden

## The map

Ten TB states, eight uneven age bands and two reachability strata. The bar chart counts
compartments per state; each should be 16 (8 ages × 2 reachability strata).

In [ ]:
model_map = pmap()
frame = pd.DataFrame(model_map.to_dicts())
per_state = frame.groupby("state", sort=False).size()
assert model_map.size == 160
assert (per_state == 16).all()
per_state.plot.bar(
    title="Compartments per TB state (8 ages × 2 reachability strata)",
    labels={"index": "state", "value": "compartments"},
)

## Demographic inputs

Per-age death rates are one summer4 table (`Data.table(...).interp("sigmoidal")`) instead of
eight summer2 functions. The lines are the UN-derived knots; `tests/test_demography.py`
checks the interpolant against the original functions at 201 half-year points.

In [ ]:
demog = load_demography()
demog.death_rates.plot(
    title="Background death rate by age band (UN WPP, Kiribati)",
    labels={"Time": "year", "value": "deaths per person per year", "age_group": "age"},
    log_y=True,
)

## Run without transmission and compare

The model runs 1850–2035 with the golden's exact parameters. The figure plots population by
age band from the port (solid) and the original (dashed); the lines should lie on top of
each other, including the step in 1950 when UN data begins.

In [ ]:
params = yaml.safe_load((GOLDEN / "no_transmission" / "params.yaml").read_text())["params"]
got = run_compartments(compile_model(), params)
gold = read_golden("no_transmission", "compartments")
got = got[gold.columns]


def by_age(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {a: df[[c for c in df.columns if f"Xage_{a}X" in c]].sum(axis=1) for a in AGE_GROUPS}
    )


compare = pd.concat(
    {"summer4": by_age(got), "summer2gen": by_age(gold)}, axis=1
).stack(level=[0, 1], future_stack=True).rename("people").reset_index()
compare.columns = ["year", "model", "age", "people"]
fig = compare.plot.line(
    x="year", y="people", color="age", line_dash="model",
    title="Population by age: summer4 port (solid) vs original (dashed)",
)
fig

## Error per compartment

Maximum absolute error per compartment over all years, in persons. Every bar should be tiny
relative to the compartment: the assertion below is the parity claim.

In [ ]:
report = parity_report(got, gold, atol=1e-6)
np.testing.assert_array_equal(got.loc[1850.0].to_numpy(), gold.loc[1850.0].to_numpy())
assert report["passes"].all(), report[~report["passes"]]
report["max_abs"].sort_values(ascending=False).head(40).plot.bar(
    title="Largest absolute errors (persons), 40 worst compartments",
    labels={"index": "compartment", "value": "max |summer4 − summer2gen|"},
    log_y=True,
)